<a href="https://colab.research.google.com/github/nayakamhrudaya/GenAI/blob/main/ChatBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# INSTALL
!pip install faiss-cpu pypdf openai

import numpy as np
import faiss
from openai import OpenAI
from pypdf import PdfReader

client = OpenAI(api_key="GENERATED_API_KEY")

# READ FILE
def read_pdf(path):
    text = ""
    pdf = PdfReader(path)
    for page in pdf.pages:
        text += page.extract_text() or ""
    return text

# CHUNK
def chunk_text(text):
    return text.split(". ")

# EMBEDDING
def embed(text):
    res = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return np.array(res.data[0].embedding, dtype=np.float32)

# BUILD INDEX
def build(chunks):
    vectors = [embed(c) for c in chunks]
    index = faiss.IndexFlatL2(len(vectors[0]))
    index.add(np.array(vectors))
    return index, chunks

# QUERY
def search(query, index, chunks):
    q = embed(query).reshape(1, -1)
    _, idx = index.search(q, 3)
    return [chunks[i] for i in idx[0]]

# GPT ANSWER
def ask(question, context):
    res = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"Context:{context}\nQuestion:{question}"}]
    )
    return res.choices[0].message.content


# -------------------------
# RUN FLOW (COLAB STYLE)
# -------------------------
text = "AI is used in chatbots. ML is subset of AI."

chunks = chunk_text(text)
index, chunks = build(chunks)

while True:
    q = input("Ask: ")
    if q == "exit":
        break

    context = search(q, index, chunks)
    print(ask(q, context))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 12.2 MB/s eta 0:00:00
Ask: hi
Hello! How can I assist you today?
Ask: what is vector db'
A vector database is a type of database designed to efficiently store, retrieve, and manage vector embeddings generated by machine learning models. These embeddings represent data points in a high-dimensional space, allowing for tasks such as similarity search, nearest neighbor search, and clustering.

Vector databases are particularly useful in applications involving natural language processing, image recognition, and recommendation systems, where data can be represented as vectors. They often provide mechanisms to perform fast queries based on the similarity between vectors, enabling advanced features like semantic search and real-time recommendations.

Some popular vector databases include Pinecone, Weaviate, and Faiss. These systems are optimized for handling